# Demo: corpus MuseTrainer con finite HMM y HDP-HMM

Este notebook clasifica la biblioteca `musetrainer/library`, ejecuta el análisis en lote con ambos modelos y muestra tablas y figuras de comparación.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

LIBRARY_DIR = PROJECT_ROOT / 'external' / 'library' / 'scores'
OUTPUT_DIR = PROJECT_ROOT / 'artifacts' / 'outputs' / 'musetrainer_corpus_both'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('LIBRARY_DIR =', LIBRARY_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)

In [ ]:
from src.analysis.library_batch import analyze_library

# Para una prueba rapida, cambia LIMIT a un numero pequeno como 8 o 12.
# Para el corpus completo, deja LIMIT = None.
LIMIT = None
OBS = 'pitch_class'
MODEL = 'both'
K = 12
ITERS = 30
BURN_IN = 15
SEED = 7

In [ ]:
outputs = analyze_library(
    library_dir=LIBRARY_DIR,
    output_dir=OUTPUT_DIR,
    obs_type=OBS,
    model=MODEL,
    limit=LIMIT,
    hdp_params={
        'n_states': K,
        'n_iters': ITERS,
        'burn_in': BURN_IN,
        'seed': SEED,
    },
)
outputs

In [ ]:
import pandas as pd

catalog = pd.read_csv(OUTPUT_DIR / 'catalog' / 'catalog.csv')
analysis = pd.read_csv(OUTPUT_DIR / 'analysis' / 'analysis.csv')

print('Obras catalogadas:', len(catalog))
print('Obras con error de parseo:', int((catalog['error'].fillna('') != '').sum()))
display(catalog[['title', 'composer', 'period', 'form', 'difficulty_bucket', 'estimated_key', 'error']].head(12))
display(analysis.head(12))

In [ ]:
report_path = OUTPUT_DIR / 'analysis' / 'analysis_report.md'
if report_path.exists():
    print(report_path.read_text())

In [ ]:
from IPython.display import Image, display

figure_paths = [
    OUTPUT_DIR / 'catalog' / 'figures' / 'catalog_by_composer.png',
    OUTPUT_DIR / 'catalog' / 'figures' / 'catalog_by_difficulty.png',
    OUTPUT_DIR / 'analysis' / 'figures' / 'compare_log_likelihood_scatter.png',
    OUTPUT_DIR / 'analysis' / 'figures' / 'compare_log_likelihood_boxplot.png',
    OUTPUT_DIR / 'analysis' / 'figures' / 'compare_state_counts_scatter.png',
    OUTPUT_DIR / 'analysis' / 'figures' / 'composer_state_complexity.png',
]

for figure_path in figure_paths:
    if figure_path.exists():
        print(figure_path.name)
        display(Image(filename=str(figure_path)))

In [ ]:
COMMAND = (
    'python -m src.cli.library_analysis '
    f'--library-dir {LIBRARY_DIR} '
    f'--model {MODEL} '
    f'--obs {OBS} '
    f'--K {K} '
    f'--iters {ITERS} '
    f'--burn-in {BURN_IN} '
    f'--output-dir {OUTPUT_DIR}'
)
print(COMMAND)